# Creates VirtualZarr store from CESM-WACCM-G6-1.5K NetCDF, then rechunks and writes to Icechunk store on s3.

- run on an m8g.4xlarge w/ 32 workers


In [1]:
import xarray as xr
import zarr
from obstore.store import from_url
import obstore as obs
from virtualizarr import open_virtual_mfdataset, open_virtual_dataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry
from distributed import Client
import icechunk
from icechunk.xarray import to_icechunk

zarr.config.set({"async.concurrency": 128})

import warnings

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

In [2]:
client = Client(n_workers=32)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/8787/status,Workers: 32
Total threads: 32,Total memory: 60.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33721,Workers: 0
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43405,Total threads: 1
Dashboard: https://cluster-wnjzi.dask.host/jupyter/proxy/40123/status,Memory: 1.90 GiB
Nanny: tcp://127.0.0.1:44365,


2025-09-18 23:05:29,941 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f278e295e92493304646e23fe5dbdd61 initialized by task ('rechunk-merge-rechunk-transfer-f94b442b5b1d4fd4d9ce2a4e21b175c2', 1, 0, 0, 0, 1, 9, 0, 0) executed on worker tcp://127.0.0.1:35113
2025-09-18 23:06:10,113 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle da55071b6bdc1559288975ed488fae78 initialized by task ('rechunk-merge-rechunk-transfer-f94b442b5b1d4fd4d9ce2a4e21b175c2', 0, 0, 0, 0, 0, 9, 0, 0) executed on worker tcp://127.0.0.1:41641
2025-09-18 23:06:10,516 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f278e295e92493304646e23fe5dbdd61 deactivated due to stimulus 'task-finished-1758236770.511984'
2025-09-18 23:06:41,283 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d2583fbbe88011d245170660709e633c initialized by task ('rechunk-merge-rechunk-transfer-f94b442b5b1d4fd4d9ce2a4e21b175c2', 2, 0, 0, 0, 2, 9, 0, 0) executed on worker tcp://127.0.0.1:44887
2025-09-18 

In [3]:
bucket = "s3://carbonplan-srm/"
prefix = "input/tensor/CESM-G6-1.5K/netcdf"
virtual_ic_prefix = "input/tensor/CESM-WACCM-G6-1.5K/icechunk/virtual_icechunk"
ic_prefix = "input/tensor/CESM-WACCM-G6-1.5K/icechunk/icechunk"
store = from_url(bucket, region="us-west-2")
registry = ObjectStoreRegistry({bucket: store})
drop_variables = [
    "gw",
    "hyam",
    "hybm",
    "P0",
    "hyai",
    "hybi",
    "ndbase",
    "nsbase",
    "nbdate",
    "nbsec",
    "mdt",
    "date",
    "datesec",
    "time_bnds",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "co2vmr",
    "ch4vmr",
    "n2ovmr",
    "f11vmr",
    "f12vmr",
    "sol_tsi",
    "nsteph",
]
parser = HDFParser(drop_variables=drop_variables)

In [4]:
stream = obs.list_with_delimiter(store, prefix=prefix, return_arrow=True)
netcdf_list = list(stream["objects"]["path"].to_numpy())
netcdf_list.remove(prefix)
netcdf_urls = [bucket + netcdf_path for netcdf_path in netcdf_list]

In [5]:
# import re

# ensemble_members = ['001','002','003']
# grouped = {}

# for member in ensemble_members:
#     grouped[member] = [path for path in netcdf_urls
#                        if f'.{member}.' in path]

In [6]:
drop_variables = [
    "gw",
    "hyam",
    "hybm",
    "P0",
    "hyai",
    "hybi",
    "ndbase",
    "nsbase",
    "nbdate",
    "nbsec",
    "mdt",
    "date",
    "datesec",
    "time_bnds",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "co2vmr",
    "ch4vmr",
    "n2ovmr",
    "f11vmr",
    "f12vmr",
    "sol_tsi",
    "nsteph",
]

In [7]:
vds = open_virtual_dataset(
    netcdf_urls[0], registry=registry, parser=parser, drop_variables=drop_variables
)

In [ ]:
ds = xr.open_dataset(netcdf_urls[-1], engine="h5netcdf")

In [ ]:
ds.attrs["case"].rsplit(".")[-1]

In [10]:
def preprocess(ds):
    """
    get ensemble member from ds attrs filename
    """
    ensemble = ds.attrs["case"].rsplit(".")[-1]

    ds = ds.expand_dims({"ensemble_member": [ensemble]})
    return ds

In [12]:
combined_vds = open_virtual_mfdataset(
    netcdf_urls,
    registry=registry,
    parser=parser,
    preprocess=preprocess,
    combine="by_coords",
    combine_attrs="drop_conflicts",
    loadable_variables=["lat", "lev", "ilev", "time", "nbnd", "lon"],
    drop_variables=drop_variables,
    parallel="dask",
)
combined_vds

/opt/coiled/env/lib/python3.13/site-packages/zarr/core/dtype/npy/bytes.py:383: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=1)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not b

<xarray.Dataset> Size: 121GB
Dimensions:          (ensemble_member: 3, time: 18251, lat: 192, lon: 288,
                      lev: 70, ilev: 71)
Coordinates:
  * ensemble_member  (ensemble_member) object 24B '001' '002' '003'
  * lat              (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon              (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
  * lev              (lev) float64 560B 5.96e-06 9.827e-06 ... 976.3 992.6
  * ilev             (ilev) float64 568B 4.5e-06 7.42e-06 ... 985.1 1e+03
  * time             (time) object 146kB 2035-01-01 00:00:00 ... 2085-01-01 0...
Data variables:
    FLDS             (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    FSDS             (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    PRECT            (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    PS               (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    QREFHT           (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    RHREFHT          (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    TREFHT           (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    TREFHTMN         (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    TREFHTMX         (ensemble_member, time, lat, lon) float32 12GB ManifestA...
    U10              (ensemble_member, time, lat, lon) float32 12GB ManifestA...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    logname:           walkerl
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [14]:
combined_vds = combined_vds.drop_vars(["ilev", "lev"])

In [17]:
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        "s3://carbonplan-srm/",
        store=icechunk.s3_store(region="us-west-2"),
    ),
)


storage = icechunk.s3_storage(
    bucket="carbonplan-srm", prefix=virtual_ic_prefix, from_env=True
)
repo = icechunk.Repository.open_or_create(storage, config)
session = repo.writable_session("main")

In [18]:
combined_vds.vz.to_icechunk(session.store)
snapshot_id = session.commit("virtual_CESM-WACCM-G6-1.5K")
print(snapshot_id)
repo.save_config()

3XSDXRE6GGXC1JVJAHR0


# read

In [19]:
credentials = icechunk.containers_credentials(
    {
        "s3://carbonplan-srm": icechunk.s3_credentials(),
    }
)

vz_repo = icechunk.Repository.open(
    storage=storage,
    config=config,
    authorize_virtual_chunk_access=credentials,
)
vz_session = vz_repo.readonly_session("main")

In [20]:
ds = xr.open_zarr(
    vz_session.store,
    zarr_format=3,
    consolidated=False,
    chunks={},
)

ds

<xarray.Dataset> Size: 121GB
Dimensions:          (ensemble_member: 3, time: 18251, lat: 192, lon: 288)
Coordinates:
  * lon              (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
  * time             (time) object 146kB 2035-01-01 00:00:00 ... 2085-01-01 0...
  * ensemble_member  (ensemble_member) object 24B '001' '002' '003'
  * lat              (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
Data variables:
    PS               (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    U10              (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    QREFHT           (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    TREFHTMN         (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    PRECT            (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    TREFHTMX         (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    FSDS             (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    RHREFHT          (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    TREFHT           (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
    FLDS             (ensemble_member, time, lat, lon) float32 12GB dask.array<chunksize=(1, 1, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    logname:           walkerl
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [23]:
write_storage_config = icechunk.s3_storage(bucket="carbonplan-srm", prefix=ic_prefix)
write_repo = icechunk.Repository.open_or_create(write_storage_config)
write_session = write_repo.writable_session("main")

In [26]:
ds = ds.chunk({"ensemble_member": 1, "time": -1, "lat": 32, "lon": 48})
ds = ds.drop_encoding()

In [ ]:
# ds.chunk({'time':8000, 'lat':48,'lon':72}) # split lat lon chunking by factor of 4. ~100MB chunks

In [29]:
to_icechunk(ds, write_session)

In [30]:
first_snapshot = write_session.commit(
    "create spatialy chunked store: {'ensemble_member':1,'time':-1,'lat':32,'lon':48}"
)

In [31]:
first_snapshot

'XYQQQVCK6F8SQAXFCHG0'

In [33]:
ic_prefix

'input/tensor/CESM-WACCM-G6-1.5K/icechunk/icechunk'

In [32]:
virtual_ic_prefix

'input/tensor/CESM-WACCM-G6-1.5K/icechunk/virtual_icechunk'